In [1]:
import os

os.environ["AEQ_SHOW_PROGRESS"] = "FALSE"

In [3]:
import matplotlib.pyplot as plt
import pandas as pd
from aequilibrae import Project
import logging
import sys
from aequilibrae import logger
import itertools
from copy import deepcopy
from build_graphs import build_graphs
from assignments import assignment
from format_matrices import format_matrices

pth = r"C:\Users\JakeMoss\models\Arkansas\model"

proj = Project()
proj.open(pth)
net = proj.network

In [4]:
format_matrices(proj)

In [5]:
carGraph, truckGraph = build_graphs(proj)

c:\Users\JakeMoss\src\atrf2026\.venv-without-tp\Lib\site-packages\aequilibrae\paths\graph.py:225: UserWarning: Found centroids not present in the graph!
[770]
  build_compressed_graph(self, remove_dead_ends)
c:\Users\JakeMoss\src\atrf2026\.venv-without-tp\Lib\site-packages\aequilibrae\paths\graph.py:225: UserWarning: Found centroids not present in the graph!
[770]
  build_compressed_graph(self, remove_dead_ends)
c:\Users\JakeMoss\src\atrf2026\.venv-without-tp\Lib\site-packages\aequilibrae\paths\graph.py:225: UserWarning: Found centroids not present in the graph!
[770]
  build_compressed_graph(self, remove_dead_ends)
c:\Users\JakeMoss\src\atrf2026\.venv-without-tp\Lib\site-packages\aequilibrae\paths\graph.py:225: UserWarning: Found centroids not present in the graph!
[ 770 4024]
  build_compressed_graph(self, remove_dead_ends)
c:\Users\JakeMoss\src\atrf2026\.venv-without-tp\Lib\site-packages\aequilibrae\paths\graph.py:225: UserWarning: Found centroids not present in the graph!
[ 770 402

In [6]:
carGraph.network.value_counts("direction")

direction
 0    291343
 1      5711
-1      3092
Name: count, dtype: int64

In [7]:
%%time
res = assignment(proj, carGraph, truckGraph, periods=["am"])

CPU times: total: 9h 27min 40s
Wall time: 38min 17s


In [11]:
res["am"].report()

,iteration,time,rgap,alpha,warnings,beta0,beta1,beta2
0,1,32.645949,inf,1.000000,,1.000000,0.000000,0.000000
1,2,61.421713,0.064437,0.204795,,1.000000,0.000000,0.000000
2,3,91.227080,0.021064,0.256366,,1.000000,0.000000,0.000000
3,4,120.491766,0.007564,0.339729,,1.000000,0.000000,0.000000
4,5,149.381119,0.005928,0.447214,,0.489897,0.405520,0.104583
...,...,...,...,...,...,...,...,...
74,75,2178.354995,0.000013,0.072838,,0.159283,0.676245,0.164472
75,76,2207.268895,0.000012,0.073145,,0.145622,0.539598,0.314780
76,77,2236.135755,0.000011,0.075791,,0.141825,0.603043,0.255133
77,78,2264.999706,0.000012,0.044918,,0.099451,0.594697,0.305851


In [8]:
for period in ['nt','am', 'md', 'pm']:
    r = res[period]
    r.procedure_id = f"{period.upper()}_EXPERIMENT"
    r.save_results(f"{period.upper()}_EXPERIMENT")

KeyError: 'nt'

In [ ]:
with proj.results_connection as conn:
    for period in ['nt','am', 'md', 'pm']:
        df_experiment = pd.read_sql(f"""SELECT * FROM '{period.upper()}_EXPERIMENT'""", conn)
        df_converged = pd.read_sql(f"""SELECT * FROM '{period.upper()}_FULLY_CONVERGED'""", conn)

        comparison = df_experiment[["link_id","PCE_tot"]].rename(columns={"PCE_tot":"PCE_EXPERIMENT"}).merge(
        df_converged[["link_id","PCE_tot"]].rename(columns={"PCE_tot":"PCE_FULLY_CONVERGED"}),
        on="link_id")
        plt = comparison.plot.scatter(x="PCE_FULLY_CONVERGED", y="PCE_EXPERIMENT")
        plt.set_title(period.upper())
        plt.figure.savefig(f"{pth}/{period}_experiment_vs_converged.png")

In [ ]:
with proj.db_connection as conn:
    df = pd.read_sql("SELECT * FROM Results", conn)
    df = df.iloc[-8:,:]
df.to_csv(f"{pth}/all_results_report.csv", index=False)

In [ ]:
import json
import ast

convergence_all = None
for _, rec in df.iterrows():
    a = json.loads(ast.literal_eval(rec.procedure_report))
    conv = pd.DataFrame({"iter":a["convergence"]["iteration"], rec.table_name: a["convergence"]["rgap"]})
    if convergence_all is None:
        convergence_all = conv
    else:
        convergence_all = convergence_all.merge(conv, on="iter", how="outer")

In [ ]:
convergence_all.to_csv(f"{pth}/convergence_all.csv", index=False)

In [ ]:
for period in ['nt','am', 'md', 'pm']:
    convergence_all_pos = convergence_all.copy()
    converged = f"{period.upper()}_FULLY_CONVERGED"
    convergence_all_pos = convergence_all_pos[convergence_all_pos[converged] > 0]
    ax = convergence_all.plot(y=converged, x="iter")
    ax.set_yscale('log')
    ax.figure.savefig(f"{pth}/convergence_{period}_log.png")

In [ ]:
rmse = ((comparison.PCE_EXPERIMENT - comparison.PCE_FULLY_CONVERGED)**2).mean()**0.5
rmse

In [ ]:
# bridges = [4829,63054,63055,74515,74516,163174]
# bridge_links = proj.network.links.data.query('link_id in @bridges')

In [ ]:
# arrays = []
# comb_ids = []
# comb_fields = []
# for link_id in bridges:
#     for field in ["lanes_10_ab", "lanes_10_ba"]:
#         comb_ids.append(link_id)
#         comb_fields.append(field[-2:])
#         lanes = bridge_links.loc[bridge_links.link_id==link_id, field].values[0]
#         arrays.append([i/max(float(lanes), 1) for i in range(int(lanes) + 1)])
# cap_multipliers = list(itertools.product(*arrays))
# len(cap_multipliers)

In [ ]:
# for factors in cap_multipliers:
#     break
#     car_graph = deepcopy(carGraph)
#     truck_graph = deepcopy(truckGraph)
#     truck_graph = deepcopy(truckGraph)

#     for period in ['am', 'pm', 'md', 'nt']:
#         for factor, field, link_id in zip(factors, comb_fields, comb_ids):
#             if "ab" in field:
#                 car_graph.network.loc[car_graph.network.link_id == link_id, f"{period}_assncap_10_{field}"] *= factor
#                 truck_graph.network.loc[car_graph.network.link_id == link_id, f"{period}_assncap_10_{field}"] *= factor
    
#     car_graph.prepare_graph(car_graph.centroids)
#     truck_graph.prepare_graph(truck_graph.centroids)
#     res = assignment(proj, car_graph, truck_graph)
#     METRIC_COMPUTATION(RES)